# Domi: seven NoSQL databases in one hour

In the lecture we talked about architectural thinking: start from the questions the system has to answer, then
pick the storage that answers them well. This lab is the other half. To choose between a hammer and a screwdriver
you have to have held both. Here you will use seven databases on the same data and notice, with your own hands,
what each one makes easy and what it makes painful.

**Domi** is a fictional food-delivery app in Bogotá. Each database holds one part of it, and most queries
come back to the same order:

> Friday 2026-10-30, 19:42. It's raining hard in Chapinero. Valentina orders from *El Ajiaco de la Abuela*:
> 1 Ajiaco santafereño + 1 Chocolate santafereño con queso + delivery = 46,900 COP.
> Jhon Fredy picks it up at 20:03 and delivers it at 20:21. That is order `O-000042`.

**How it works.** Each cell sends a native query (Redis commands, MongoDB shell, CQL, Cypher, SQL, JSON) to a
shared server. The helpers `redis("...")`, `mongo("...")`, etc. only carry your query there; they do not rewrite it.

- First do **File > Save a copy in Drive** so you can edit.
- **Your turn** sections are for you to write. **Where it breaks** sections show a question the database handles badly.
- Everyone shares the same data. Anything you write is tagged with your nickname so you can find it.

In [ ]:
#@title Setup: fill in the three fields, then run this cell
GATEWAY = "http://localhost:8000"  #@param {type:"string"}
CLASS_CODE = "domi"  #@param {type:"string"}
NICK = "david"  #@param {type:"string"}

%pip install -q requests pandas matplotlib fastembed

import json, re
import requests, pandas as pd, matplotlib.pyplot as plt, matplotlib.dates as mdates
pd.set_option("display.max_colwidth", 80)

NICK = re.sub(r"[^a-z0-9]", "", NICK.lower()) or "anon"
GATEWAY = GATEWAY.rstrip("/")


def _send(engine, query):
    r = requests.post(f"{GATEWAY}/{engine}", data=query.encode("utf-8"), timeout=20,
                      headers={"Authorization": f"Bearer {CLASS_CODE}"})
    if r.status_code != 200:
        try:
            msg = r.json().get("detail", r.text)
        except ValueError:
            msg = r.text
        print(f"Error {r.status_code}:", msg if isinstance(msg, str) else json.dumps(msg, ensure_ascii=False)[:500])
        return None
    return r.json()


def _table(x):
    """Show a list of rows as a table; anything else as-is."""
    if isinstance(x, list) and x and all(isinstance(r, dict) for r in x):
        return pd.DataFrame(x)
    return x


def redis(q):   return _send("redis", q)
def mongo(q):   return _table(_send("mongo", q))
def neo4j(q):   return _table(_send("neo4j", q))
def influx(q):  return _table(_send("influx", q))

def scylla(q):
    r = _send("scylla", q)
    if isinstance(r, dict) and "warnings" in r:
        for w in r["warnings"]:
            print("ScyllaDB warning:", w)
        r = r["rows"]
    return _table(r) if r != [] else "(no rows)"

def elastic(q):
    r = _send("elastic", q)
    if isinstance(r, dict) and "hits" in r:
        return pd.DataFrame([{"_score": round(h["_score"] or 0, 2), **h["_source"]} for h in r["hits"]["hits"]])
    return r

def qdrant(q):
    r = _send("qdrant", q)
    if isinstance(r, dict) and isinstance(r.get("result"), dict) and "points" in r["result"]:
        return pd.DataFrame([{"score": round(p["score"], 3), **p["payload"]} for p in r["result"]["points"]])
    return r


health = requests.get(f"{GATEWAY}/health", timeout=20).json()
print(f"Nickname: {NICK}")
print("Databases:", ", ".join(f"{e} {s}" for e, s in health.items()))
print("Class code accepted." if redis("PING") else "Class code rejected: check CLASS_CODE.")

---
## 1. Key-value: Redis

Redis stores a value under a key. If you know the key, the read is instant. You cannot ask questions about the
values themselves. Domi uses it for sessions, carts, and the top-rated ranking.

Keys used here: `user:U-0001` (a hash with name and localidad) and `ranking:restaurants:rating` (a sorted set).

Get Valentina by her key.

In [ ]:
redis("HGETALL user:U-0001")

A login session that expires on its own after 30 seconds. Run the `TTL` cell a few times, then the `GET` cell.

In [ ]:
redis(f"SET session:{NICK} U-0001 EX 30")

In [ ]:
redis(f"TTL session:{NICK}")   # seconds left; run it again

In [ ]:
redis(f"GET session:{NICK}")   # after 30 s this returns None: the key is gone

Top 5 restaurants by rating. The sorted set stays ordered on every write, so reading the top is cheap.

In [ ]:
redis("ZREVRANGE ranking:restaurants:rating 0 4 WITHSCORES")

### Your turn
Build a cart: add 2 empanadas, then one more, then read the whole cart. Add another item too.

In [ ]:
redis(f"HSET cart:{NICK} empanada 2")
redis(f"HINCRBY cart:{NICK} empanada 1")
redis(f"HGETALL cart:{NICK}")

### Where it breaks
"Find all users who live in Suba." There is no command for that, because Redis only looks things up by key.
You would have needed to create a key like `localidad:suba:users` in advance. Also, on a shared server some
commands are disabled:

In [ ]:
redis("DEL user:U-0001")

---
## 2. Document: MongoDB

MongoDB stores each record as a document, shaped the way the application uses it: nested objects and arrays,
no fixed schema. Domi keeps orders with their items inside, and restaurants with their menus inside.

Order O-000042, whole. The items are inside the order, with the price as it was paid.

In [ ]:
mongo('orders.findOne({_id: "O-000042"})')

If El Ajiaco raises its prices tomorrow, Valentina's old order does not change. Embedding is a design decision.

Query inside an array: every order that contains an ajiaco.

In [ ]:
mongo('orders.find({"items.name": "Ajiaco santafereño"}, {user_id: 1, total_cop: 1, placed_at: 1})')

Top 5 dishes: unwind the items, group by name, sort.

In [ ]:
mongo('''orders.aggregate([
  {$unwind: "$items"},
  {$group: {_id: "$items.name", sold: {$sum: "$items.qty"}}},
  {$sort: {sold: -1}},
  {$limit: 5}
])''')

### Your turn
Open your own restaurant, with a field no other restaurant has. MongoDB accepts it. Then find it.

In [ ]:
mongo(f'restaurants.insertOne({{name: "Donde {NICK}", cuisine: "Fusion", author: "{NICK}", parking: true}})')
mongo(f'restaurants.find({{author: "{NICK}"}})')

### Where it breaks
"Valentina's orders, with each restaurant's rating." The rating is in another collection, so you need `$lookup`,
which is a join. It works, but needing it often means the data was split the wrong way for this question.

In [ ]:
mongo('''orders.aggregate([
  {$match: {user_id: "U-0001"}},
  {$lookup: {from: "restaurants", localField: "restaurant_id", foreignField: "_id", as: "restaurant"}},
  {$project: {total_cop: 1, rating: {$first: "$restaurant.rating"}}}
])''')

In [ ]:
mongo('orders.deleteOne({_id: "O-000042"})')

---
## 3. Wide-column: ScyllaDB (Cassandra's CQL)

ScyllaDB handles very high write volume and fast reads, but only when you query by the partition key. You design
one table per question, and storing the same data more than once is expected. CQL looks like SQL but does not
behave like it. Domi uses it for courier GPS pings and order history per user.

| Table | Partition key | Sorted by |
|---|---|---|
| `courier_pings_by_day` | `(courier_id, day)` | `ts DESC` |
| `orders_by_user` | `(user_id)` | `placed_at DESC` |

Jhon Fredy's GPS pings during order O-000042. Times are UTC: 20:03 in Bogotá is 01:03 UTC.

In [ ]:
route = scylla("""SELECT ts, lat, lon FROM domi.courier_pings_by_day
WHERE courier_id = 'C-01' AND day = '2026-10-30'
  AND ts >= '2026-10-31 01:03:00+0000' AND ts <= '2026-10-31 01:21:00+0000'""")
route.head()

In [ ]:
# the route: one partition, already sorted by time
plt.figure(figsize=(6, 5))
plt.scatter(route.lon, route.lat, c=range(len(route))[::-1], cmap="viridis", s=18)
plt.annotate("  pickup: El Ajiaco, 20:03", (route.lon.iloc[-1], route.lat.iloc[-1]))
plt.annotate("delivered: Valentina, 20:21", (route.lon.iloc[0], route.lat.iloc[0]), ha="right", va="bottom")
plt.xticks([]); plt.yticks([])
plt.title("C-01, order O-000042: one ping every 30 s"); plt.xlabel("lon"); plt.ylabel("lat"); plt.show()

Valentina's order history: one partition read, already sorted newest first.

In [ ]:
scylla("SELECT * FROM domi.orders_by_user WHERE user_id = 'U-0001'")

Insert the same primary key twice with different totals. There is no error: the second write replaces the first.

In [ ]:
scylla(f"INSERT INTO domi.orders_by_user (user_id, placed_at, order_id, restaurant_name, total_cop) VALUES ('S-{NICK}', '2026-11-13 20:00:00+0000', 'O-{NICK}', 'Test', 10000)")
scylla(f"INSERT INTO domi.orders_by_user (user_id, placed_at, order_id, restaurant_name, total_cop) VALUES ('S-{NICK}', '2026-11-13 20:00:00+0000', 'O-{NICK}', 'Test', 99999)")
scylla(f"SELECT * FROM domi.orders_by_user WHERE user_id = 'S-{NICK}'")

### Your turn
You are a courier now. Insert a ping for `C-<you>` and read your partition back. Add a few more pings.

In [ ]:
scylla(f"INSERT INTO domi.courier_pings_by_day (courier_id, day, ts, lat, lon) VALUES ('C-{NICK}', '2026-11-13', '2026-11-13 20:00:00+0000', 4.6486, -74.0628)")
scylla(f"SELECT * FROM domi.courier_pings_by_day WHERE courier_id = 'C-{NICK}' AND day = '2026-11-13'")

### Where it breaks
"All orders from El Ajiaco." Orders are partitioned by user, so ScyllaDB refuses to scan for a restaurant.
The fix is another table, `orders_by_restaurant`, with the same orders stored a second time.

In [ ]:
scylla("SELECT * FROM domi.orders_by_user WHERE restaurant_name = 'El Ajiaco de la Abuela'")

---
## 4. Graph: Neo4j

Neo4j stores nodes and the relationships between them. A question that needs many JOINs in SQL becomes a single
pattern. Domi uses it for recommendations and the referral program.

`(User)-[:PLACED]->(Order)-[:CONTAINS]->(Dish)<-[:SERVES]-(Restaurant)`, `(Courier)-[:DELIVERED]->(Order)`, `(User)-[:REFERRED]->(User)`

Order O-000042 as a pattern. (Neo4j Browser can draw the same query as a picture.)

In [ ]:
neo4j("""MATCH (u:User)-[:PLACED]->(o:Order {order_id: 'O-000042'})-[:CONTAINS]->(d:Dish)<-[:SERVES]-(r:Restaurant),
      (c:Courier)-[:DELIVERED]->(o)
RETURN u.name AS user, o.order_id AS order, d.name AS dish, r.name AS restaurant, c.name AS courier""")

Recommendations: what do other people who ordered ajiaco also order, that Valentina hasn't tried?

In [ ]:
neo4j("""MATCH (v:User {user_id: 'U-0001'})-[:PLACED]->(:Order)-[:CONTAINS]->(a:Dish {name: 'Ajiaco santafereño'})
MATCH (a)<-[:CONTAINS]-(:Order)<-[:PLACED]-(other:User)-[:PLACED]->(:Order)-[:CONTAINS]->(rec:Dish)
WHERE other <> v AND NOT (v)-[:PLACED]->(:Order)-[:CONTAINS]->(rec)
RETURN rec.name AS dish, count(DISTINCT other) AS people
ORDER BY people DESC LIMIT 5""")

The referral chain: everyone Valentina brought in, up to 3 hops away.

In [ ]:
neo4j("""MATCH path = (:User {user_id: 'U-0001'})-[:REFERRED*1..3]->(u:User)
RETURN u.name AS user, length(path) AS hops""")

### Your turn
Join the chain: create yourself, have Camila refer you, then run the referral query above again. You should appear 3 hops from Valentina.

In [ ]:
neo4j(f"""MERGE (me:User {{user_id: 'S-{NICK}'}}) SET me.name = '{NICK}'
WITH me MATCH (camila:User {{user_id: 'U-0003'}})
MERGE (camila)-[:REFERRED]->(me)
RETURN camila.name AS referrer, me.name AS you""")

### Where it breaks
This one is a contrast with SQL rather than a failure. The referral query above would be a recursive query in SQL
(`WITH RECURSIVE ... JOIN ... JOIN ...`); here it is one arrow with `*1..3`. Deleting on the shared graph is disabled:

In [ ]:
neo4j("MATCH (u:User {user_id: 'U-0001'}) DETACH DELETE u")

---
## 5. Time-series: InfluxDB 3 (SQL)

InfluxDB stores measurements over time and is built for queries over time windows and aggregates. It uses SQL,
so what's new here is the model, not the language. Domi uses it for the operations dashboard, where one pattern
stands out: when it rains in Bogotá, orders go up.

| Table | Tags (filter and group by) | Fields (what is measured) |
|---|---|---|
| `orders` | `localidad`, `restaurant_id` | `total_cop` (one point per order) |
| `weather` | `localidad` | `rain_mm`, `temp_c` (hourly) |

Times are UTC. Bogotá is UTC-5.

Friday 2026-10-30 in Chapinero: orders per 15 minutes (`date_bin`), and the rain.

In [ ]:
orders = influx("""SELECT date_bin(INTERVAL '15 minutes', time) AS t, count(*) AS orders
FROM orders
WHERE localidad = 'Chapinero' AND time >= '2026-10-30T11:00:00Z' AND time < '2026-10-31T05:00:00Z'
GROUP BY 1 ORDER BY 1""")
rain = influx("""SELECT time AS t, rain_mm FROM weather
WHERE localidad = 'Chapinero' AND time >= '2026-10-30T11:00:00Z' AND time < '2026-10-31T05:00:00Z'
ORDER BY time""")
orders.tail()

In [ ]:
# orders vs. rain, in Bogotá time
bog = lambda s: pd.to_datetime(s) - pd.Timedelta(hours=5)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(bog(orders.t), orders.orders, width=0.009, color="tab:orange", label="orders / 15 min")
ax2 = ax.twinx()
ax2.plot(bog(rain.t), rain.rain_mm, color="tab:blue", marker="o", label="rain mm/h")
ax.set_ylabel("orders"); ax2.set_ylabel("rain (mm/h)")
ax.set_title("Chapinero, Friday 2026-10-30 (Bogotá time)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
fig.legend(loc="upper left"); plt.show()

Orders by hour of day over two weeks (Bogotá time). Look for the lunch and dinner peaks.

In [ ]:
influx("""SELECT extract(hour FROM time - INTERVAL '5 hours') AS hour, count(*) AS orders
FROM orders GROUP BY 1 ORDER BY 1""").plot.bar(x="hour", y="orders", figsize=(10, 3), legend=False);

### Your turn
Report the weather where you are, in line protocol: `table,tag=...,tag=... field=...,field=...`. Then query it back.

In [ ]:
influx(f"weather,localidad=Chapinero,author={NICK} rain_mm=12.5,temp_c=9.0")
influx(f"SELECT * FROM weather WHERE author = '{NICK}'")

### Where it breaks
"Show me the items of order O-000042." They aren't here. There is no nested data and no lookup by ID.

In [ ]:
influx("SELECT * FROM orders WHERE order_id = 'O-000042'")

---
## 6. Search: Elasticsearch

Elasticsearch does full-text search: it tolerates typos, accents and capitalization, and ranks the best match
first. Domi uses it for the search bar, over the `dishes` index. Each query has the form `METHOD /path` on the
first line, then the JSON body.

Typos still match. Try `ajiacco`, `AJIACO`, `santafereno`.

In [ ]:
elastic("""GET /dishes/_search
{"query": {"multi_match": {"query": "ajiacco", "fields": ["name", "description"], "fuzziness": "AUTO"}},
 "_source": ["name", "restaurant_name"], "size": 5}""")

What the engine actually stores: tokens, lowercased, with the accents removed.

In [ ]:
r = elastic("""GET /dishes/_analyze
{"text": "Ajiaco Santafereño con Guascas"}""")
[t["token"] for t in r["tokens"]]

Text plus filters: soups in Chapinero for 30,000 COP or less.

In [ ]:
elastic("""GET /dishes/_search
{"query": {"bool": {
   "must":   {"match": {"tags": "soup"}},
   "filter": [{"term": {"localidad": "Chapinero"}}, {"range": {"price_cop": {"lte": 30000}}}]}},
 "_source": ["name", "restaurant_name", "price_cop"]}""")

### Your turn
Add your own dish to the menu, then find it with a typo.

In [ ]:
MY_DISH = "Obleas con arequipe"   # change me
MY_TYPO = "obleas con arekipe"    # and this
elastic(f"""POST /dishes/_doc?refresh=true
{{"name": "{MY_DISH}", "description": "Made by {NICK}", "author": "{NICK}", "price_cop": 25000}}""")
elastic(f"""GET /dishes/_search
{{"query": {{"match": {{"name": {{"query": "{MY_TYPO}", "fuzziness": "AUTO"}}}}}}}}""")

### Where it breaks
"Something warm for a cold rainy night." The answer you'd expect is ajiaco, but no dish description contains
those words, so it doesn't come up near the top. We'll use the same phrase in the next section.

In [ ]:
elastic("""GET /dishes/_search
{"query": {"multi_match": {"query": "something warm for a cold rainy night", "fields": ["name", "description", "tags"]}},
 "_source": ["name", "restaurant_name"], "size": 5}""")

---
## 7. Vector: Qdrant

Qdrant searches by meaning instead of by words. A model turns text into a list of numbers, and the database finds
the closest lists. Domi would use it for "I don't know what I want, something warm for a rainy night."

Every dish was turned into 384 numbers by the model `all-MiniLM-L6-v2`. You use the same model here for your query.

In [ ]:
from fastembed import TextEmbedding
model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")   # ~90 MB, first run takes a few seconds

def embed(text):
    return [round(float(x), 4) for x in next(model.embed([text]))]

What "ajiaco" looks like to the database.

In [ ]:
v = embed("ajiaco")
print(len(v), "numbers:", v[:8], "...")

The phrase that failed in Elasticsearch, now searched by meaning.

In [ ]:
v = embed("something warm for a cold rainy night")
qdrant(f"""POST /collections/dishes/points/query
{{"query": {json.dumps(v)}, "limit": 5, "with_payload": true}}""")

Same search, restricted to Chapinero and under 30,000 COP (a payload filter).

In [ ]:
qdrant(f"""POST /collections/dishes/points/query
{{"query": {json.dumps(v)}, "limit": 5, "with_payload": true,
  "filter": {{"must": [{{"key": "localidad", "match": {{"value": "Chapinero"}}}},
                      {{"key": "price_cop", "range": {{"lte": 30000}}}}]}}}}""")

### Your turn
Search for your own craving: "something light after the gym", "a hangover breakfast", "algo para compartir".

In [ ]:
CRAVING = "a hangover breakfast"   # change me
qdrant(f"""POST /collections/dishes/points/query
{{"query": {json.dumps(embed(CRAVING))}, "limit": 5, "with_payload": true}}""")

### Where it breaks
Search for an exact code, `O-000042`. You still get 5 "nearest" dishes with scores that look meaningful, and they
mean nothing. Vectors are bad at exact matches, which is why real applications combine keyword and vector search.

In [ ]:
qdrant(f"""POST /collections/dishes/points/query
{{"query": {json.dumps(embed("O-000042"))}, "limit": 5, "with_payload": true}}""")

---
## Wrap-up: which database answers which question?

| Domi question | Database | Why |
|---|---|---|
| Who is logged in? What's in my cart? | Redis (key-value) | You know the key and need the answer immediately |
| Show order O-000042 with its items | MongoDB (document) | One order is one document, read whole |
| Where was the courier every 30 s? My order history? | ScyllaDB (wide-column) | Many writes, read by partition key |
| Recommendations, referral chains | Neo4j (graph) | The relationships are the data |
| Do orders go up when it rains? | InfluxDB (time-series) | Aggregates over time windows |
| The search bar: "ajiacco" | Elasticsearch (search) | Text, typos, ranking |
| "Something warm for a rainy night" | Qdrant (vector) | Meaning, not words |

None of these is the best database. Each one is good at the questions it was designed for and awkward at the
rest, which is what the "Where it breaks" sections showed. Start from the question, then choose the tool.